# Step 5 – Environmental optimisation: CO₂ emission costs (oemof)
**Sweden · SE3 2019 · same five producers and the capacities from Step 4 · demand + DHNx network losses · no storage**

Step 5 keeps the energy system of Step 4 (same producers, **same installed capacities**, same demand including the network losses) and adds a cost for every tonne of CO₂ emitted. Because the capacities are now fixed, the model only decides the **hourly dispatch**. The objective function is unchanged: oemof still minimises total cost, which now includes the emission cost.

| Run | Fuel / electricity price | Variable O&M | CO₂ cost |
|---|---|---|---|
| **Reference (Step 4)** | ✓ | ✓ | – |
| **Scenario A – cost + emissions** | ✓ | ✓ | ✓ 134 €/t |
| **Scenario B – emissions only** | 0 | 0 | ✓ 134 €/t |

Sensitivity analyses (section 9): (1) the carbon price in scenario A from 0 to 500 €/t; (2) biogenic CO₂ counted as zero, as in IPCC inventory practice and EU ETS.

| | |
|---|---|
| **Reads** | `data/step4_loss/oemof_input_with_losses.csv` (demand + losses, prices) · `data/step3/technology_parameters.csv`, `carriers.csv` · `results/step4_loss/unit_summary.csv` (fixed capacities), `dispatch.csv` |
| **Writes** | `results/step5_emissions/` (tables, figures, Excel file) |

Run Step 2, Step 3 and `step4_loss.ipynb` first. Units: MW, MWh, €, tCO₂.

## 0. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from oemof.solph import Bus, EnergySystem, Flow, Model, processing
from oemof.solph.components import Converter, Sink, Source
from helpers import DATA, RESULTS, SOLVER, SOLVER_OPTIONS, plot_dispatch, UNIT_COLORS

STEP3_IN, STEP4_IN, STEP4_OUT = DATA / "step3", DATA / "step4_loss", RESULTS / "step4_loss"
OUT = RESULTS / "step5_emissions"
OUT.mkdir(parents=True, exist_ok=True)

# Scenario colours used in all comparison figures; units keep their Step 3 colours.
SCEN_COLORS = {"Step 4": "#9E9E9E", "A: cost + CO2": "#B2182B", "B: CO2 only": "#2166AC"}
print("Solver:", SOLVER)

## 1. Inputs from Steps 3 and 4
- **Hourly data** (demand including network losses, SE3 electricity price, biomethane price): the file Step 4 saved.
- **Technology data** (efficiency/COP, variable O&M, CAPEX, lifetime): unchanged from Step 3.
- **Installed capacities: fixed at the Step 4 result.** They are no longer decision variables. Their annualised CAPEX is a sunk cost, identical in every scenario, so it does not affect the dispatch; it is only added to the total cost for reporting.

In [ ]:
for f in [STEP4_IN / "oemof_input_with_losses.csv", STEP3_IN / "technology_parameters.csv",
          STEP3_IN / "carriers.csv", STEP4_OUT / "unit_summary.csv", STEP4_OUT / "dispatch.csv"]:
    assert f.exists(), f"Missing {f.relative_to(DATA.parent)}: run Step 3 and step4_loss.ipynb first"

data = pd.read_csv(STEP4_IN / "oemof_input_with_losses.csv", index_col=0, parse_dates=True)
data.index = pd.DatetimeIndex(data.index, freq="h")      # oemof needs a regular hourly index
tech = pd.read_csv(STEP3_IN / "technology_parameters.csv", index_col=0)
carriers_s3 = pd.read_csv(STEP3_IN / "carriers.csv", index_col=0)
summary_s4 = pd.read_csv(STEP4_OUT / "unit_summary.csv", index_col=0)
dispatch_s4 = pd.read_csv(STEP4_OUT / "dispatch.csv", index_col=0, parse_dates=True)

units = tech.index.tolist()
assert summary_s4.index.drop("TOTAL").tolist() == units, "Step 4 summary and technology table list different units"
assert len(data) == 8760 and (dispatch_s4.index == data.index).all()

DEMAND = "heat demand incl. losses"
q_tot = data[DEMAND]
cap = summary_s4.loc[units, "capacity [MW]"].astype(float)            # fixed capacities [MW]
capex_annual = summary_s4.loc[units, "annualised capex [EUR]"].astype(float)   # sunk, EUR/yr

print(f"Demand incl. losses: peak {q_tot.max():.4f} MW | annual {q_tot.sum():,.1f} MWh")
print(f"Fixed installed capacity: {cap.sum():.4f} MW  (difference to peak: {1000 * (cap.sum() - q_tot.max()):+.4f} kW)")
display(pd.DataFrame({"fixed capacity [MW]": cap, "site limit [MW]": tech["max_capacity_MW"],
                      "input": tech["input"], "efficiency / COP": tech["efficiency"],
                      "var O&M [EUR/MWh heat]": tech["var_om_eur_per_MWh"],
                      "annualised CAPEX [EUR/yr]": capex_annual}).round(4))

## 2. Emission factors and carbon price
🔧 **ASSUMPTIONS** (state and justify in the report):

- **Carbon price: 134 €/tCO₂**, from the IEA. Applied equally to every tonne, from every unit.
- **Biogenic CO₂ is counted.** CO₂ from burning wood pellets and biomethane is counted at the stack, like fossil CO₂: the regrowth of the biomass happens elsewhere and later, while the emission happens in the city now. This deliberately differs from IPCC national-inventory practice and the EU ETS, where biogenic CO₂ counts as zero; that convention is tested as a sensitivity in section 9.2.
- **Electricity:** Swedish electricity production mix, ≈ 20 g CO₂/kWh.
- **Direct (combustion / generation) emissions only.** Upstream fuel chains and the manufacture and end-of-life of the plants are not included.

| Carrier | Factor | Source |
|---|---|---|
| Electricity | ≈ 20 g CO₂/kWh = 0.020 tCO₂/MWh | Uniper, *El och klimat*: https://www.uniper.energy/sv/sverige/en-elforsorjning-som-levererar/elektrifieringen/el-och-klimat |
| Wood pellets | 112,000 kg CO₂/TJ × 0.0036 TJ/MWh = 0.403 tCO₂/MWh fuel | IPCC 2006 Guidelines, Vol. 2, Ch. 2, Table 2.2, "Wood/Wood Waste": https://www.ipcc-nggip.iges.or.jp/public/2006gl/pdf/2_Volume2/V2_2_Ch2_Stationary_Combustion.pdf |
| Biomethane | 54,600 kg CO₂/TJ × 0.0036 TJ/MWh = 0.197 tCO₂/MWh fuel | Same table, "Other Biogas" |

The emission cost of a unit per MWh of heat is **carbon price × emission factor / efficiency**. For the heat pumps the efficiency is the COP (3.1), so their electricity emissions are divided by 3.1. The recovered heat they use is emission-free.

In [ ]:
CARBON_PRICE = 134.0                     # EUR/tCO2, IEA
TJ_PER_MWH = 0.0036

emission_factors = pd.DataFrame([
    ["electricity", 20 / 1000,                    False, "Uniper, El och klimat: Swedish production mix ~20 g CO2/kWh"],
    ["biomass",     112_000 * TJ_PER_MWH / 1000,  True,  "IPCC 2006 Vol.2 Ch.2 Table 2.2, Wood/Wood Waste 112,000 kg/TJ"],
    ["gas",         54_600 * TJ_PER_MWH / 1000,   True,  "IPCC 2006 Vol.2 Ch.2 Table 2.2, Other Biogas 54,600 kg/TJ (biomethane)"],
], columns=["carrier", "ef_tCO2_per_MWh_input", "biogenic", "source"]).set_index("carrier")

# Sensitivity 9.2: biogenic CO2 counted as zero (IPCC inventory / EU ETS convention)
ef_biogenic_zero = emission_factors.copy()
ef_biogenic_zero.loc[ef_biogenic_zero["biogenic"], "ef_tCO2_per_MWh_input"] = 0.0
display(emission_factors)

# Variable cost per MWh of heat, per unit (annual means, electricity at the mean SE3 price)
mean_price = {"electricity": data["el_spot_price"].mean(), "gas": data["gas price"].mean(),
              "biomass": carriers_s3.loc["biomass", "price_eur_per_MWh"]}
eff = tech["efficiency"]
fuel_eur = tech["input"].map(mean_price) / eff
om_eur = tech["var_om_eur_per_MWh"]
co2_t = tech["input"].map(emission_factors["ef_tCO2_per_MWh_input"]) / eff
merit = pd.DataFrame({
    "tCO2 per MWh heat": co2_t,
    "fuel/electricity [EUR/MWh heat]": fuel_eur,
    "var O&M [EUR/MWh heat]": om_eur,
    "CO2 cost [EUR/MWh heat]": CARBON_PRICE * co2_t,
})
merit["Step 4 (fuel + O&M)"] = fuel_eur + om_eur
merit["A: cost + CO2"] = fuel_eur + om_eur + CARBON_PRICE * co2_t
merit["B: CO2 only"] = CARBON_PRICE * co2_t
display(merit.round(3))
print("Electricity varies hourly (SE3); the values above use the annual mean "
      f"{mean_price['electricity']:.1f} EUR/MWh. The model itself uses the hourly prices.")

ax = merit[["Step 4 (fuel + O&M)", "A: cost + CO2", "B: CO2 only"]].plot.barh(
    figsize=(11, 4.5), width=0.8, color=[SCEN_COLORS["Step 4"], SCEN_COLORS["A: cost + CO2"], SCEN_COLORS["B: CO2 only"]])
ax.invert_yaxis(); ax.set_xlabel("Variable cost [EUR/MWh heat] (annual mean)")
ax.set_title("Merit order: what each unit costs to run per MWh of heat, per scenario")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False); ax.grid(axis="x", alpha=0.3)
plt.tight_layout(); plt.savefig(OUT / "merit_order.png", dpi=150, bbox_inches="tight"); plt.show()

## 3. Dispatch model with fixed capacities
Written in the style of the oemof tutorial (`district_heating_supply_2`) and identical in structure to the Step 3/4 model in `helpers.py`, with two differences:
1. **Capacities are fixed** (`nominal_capacity` = the Step 4 value) instead of `Investment` objects, so only the hourly dispatch is optimised.
2. **The CO₂ cost is put on the fuel and electricity sources**: every MWh of fuel or electricity bought costs *price + carbon price × emission factor*. This is equivalent to adding *carbon price × EF / efficiency* to each unit's variable cost per MWh of heat.

Decision variables: hourly heat output of each unit [MW] (and the corresponding fuel/electricity input).
Objective: minimise Σₜ [fuel/electricity cost + variable O&M + CO₂ cost], over all 8760 hours.
Constraints: hourly heat balance (supply = demand + losses), output ≤ fixed capacity, input = output / efficiency (heat pumps: electricity = heat / COP, recovered heat = heat × (COP − 1)/COP).

A safety source **"unserved heat"** (10,000 €/MWh) is added so the problem can never become infeasible through rounding: the fixed capacities equal the peak demand almost exactly. It must be ≈ 0 in every result, and this is checked.

In [ ]:
UNSERVED_PENALTY = 10_000.0      # EUR/MWh, far above any real cost


def solve_dispatch(ef_table, co2_price, include_fuel=True, include_om=True):
    # Build and solve the fixed-capacity dispatch model. Returns (model, results, parts).
    f = 1.0 if include_fuel else 0.0
    om = 1.0 if include_om else 0.0
    ef = ef_table["ef_tCO2_per_MWh_input"]

    es = EnergySystem(timeindex=data.index, infer_last_interval=True)

    heat_bus = Bus(label="heat network")
    gas_bus = Bus(label="gas network")
    biomass_bus = Bus(label="biomass network")
    electricity_bus = Bus(label="electricity network")
    es.add(heat_bus, gas_bus, biomass_bus, electricity_bus)

    # Fuel and electricity sources: price (x f) + CO2 cost per MWh of input
    es.add(Source(label="gas source", outputs={gas_bus: Flow(
        variable_costs=f * data["gas price"] + co2_price * ef["gas"])}))
    es.add(Source(label="biomass source", outputs={biomass_bus: Flow(
        variable_costs=f * carriers_s3.loc["biomass", "price_eur_per_MWh"] + co2_price * ef["biomass"])}))
    es.add(Source(label="electricity source", outputs={electricity_bus: Flow(
        variable_costs=f * data["el_spot_price"] + co2_price * ef["electricity"])}))

    # Demand incl. network losses (fixed profile) and the safety source
    heat_sink = Sink(label="heat sink", inputs={heat_bus: Flow(
        nominal_capacity=q_tot.max(), fix=q_tot / q_tot.max())})
    unserved = Source(label="unserved heat", outputs={heat_bus: Flow(variable_costs=UNSERVED_PENALTY)})
    es.add(heat_sink, unserved)

    carrier_bus = {"gas": gas_bus, "biomass": biomass_bus, "electricity": electricity_bus}
    unit_parts = {}
    for unit, t in tech.iterrows():
        out_flow = Flow(nominal_capacity=float(cap[unit]), variable_costs=om * t["var_om_eur_per_MWh"])
        in_bus = carrier_bus[t["input"]]
        if t["technology_type"] == "heat_pump":
            cop = float(t["efficiency"])
            source_bus = Bus(label=f"{unit} heat source bus")           # free recovered heat
            es.add(source_bus, Source(label=f"{unit} heat source", outputs={source_bus: Flow()}))
            comp = Converter(label=unit, inputs={in_bus: Flow(), source_bus: Flow()},
                             outputs={heat_bus: out_flow},
                             conversion_factors={in_bus: 1 / cop, source_bus: (cop - 1) / cop, heat_bus: 1})
        else:
            comp = Converter(label=unit, inputs={in_bus: Flow()}, outputs={heat_bus: out_flow},
                             conversion_factors={heat_bus: float(t["efficiency"])})
        es.add(comp)
        unit_parts[unit] = (comp, in_bus)

    model = Model(es)
    model.solve(solver=SOLVER, solve_kwargs={"tee": False}, cmdline_options=SOLVER_OPTIONS)
    sr = model.solver_results
    status = (sr.solver.termination_condition if hasattr(sr, "solver")
              else sr.get("termination_condition", "n/a"))
    results = processing.results(model)
    parts = dict(heat_bus=heat_bus, heat_sink=heat_sink, unserved=unserved, units=unit_parts,
                 status=str(status), objective=model.objective())
    return model, results, parts


def dispatch_of(results, parts):
    # Hourly heat per unit [MW], unserved heat and demand.
    hb, idx = parts["heat_bus"], data.index
    d = pd.DataFrame({u: results[(c, hb)]["sequences"]["flow"].reindex(idx) for u, (c, _) in parts["units"].items()})
    d["unserved heat"] = results[(parts["unserved"], hb)]["sequences"]["flow"].reindex(idx)
    d["demand"] = results[(hb, parts["heat_sink"])]["sequences"]["flow"].reindex(idx)
    return d


def evaluate(results, parts, ef_table, co2_price=CARBON_PRICE):
    # Per-unit results. Costs always use the REAL prices, so every scenario is judged on the
    # same economic basis even if the optimisation ignored some costs (scenario B).
    hb, idx = parts["heat_bus"], data.index
    ef = ef_table["ef_tCO2_per_MWh_input"]
    rows = {}
    for u, (comp, in_bus) in parts["units"].items():
        heat = results[(comp, hb)]["sequences"]["flow"].reindex(idx)
        fuel_in = results[(in_bus, comp)]["sequences"]["flow"].reindex(idx)
        carrier = tech.loc[u, "input"]
        price = (data["el_spot_price"] if carrier == "electricity" else
                 data["gas price"] if carrier == "gas" else carriers_s3.loc["biomass", "price_eur_per_MWh"])
        co2 = float((fuel_in * ef[carrier]).sum())
        rows[u] = {
            "capacity [MW]": cap[u],
            "heat [MWh]": heat.sum(),
            "share of heat [%]": 0.0,
            "full-load hours [h]": heat.sum() / cap[u] if cap[u] > 1e-6 else 0.0,
            "hours at max capacity [h]": int((heat >= 0.999 * cap[u]).sum()) if cap[u] > 1e-6 else 0,
            "fuel/electricity cost [EUR]": float((fuel_in * price).sum()),
            "var O&M cost [EUR]": heat.sum() * tech.loc[u, "var_om_eur_per_MWh"],
            "operating cost [EUR]": 0.0,
            "annualised capex [EUR]": capex_annual[u],
            "CO2 [t]": co2,
            "CO2 cost [EUR]": co2 * co2_price,
        }
    df = pd.DataFrame(rows).T
    df["share of heat [%]"] = 100 * df["heat [MWh]"] / df["heat [MWh]"].sum()
    df["operating cost [EUR]"] = df["fuel/electricity cost [EUR]"] + df["var O&M cost [EUR]"]
    df["economic cost [EUR]"] = df["operating cost [EUR]"] + df["annualised capex [EUR]"]
    df["economic cost + CO2 cost [EUR]"] = df["economic cost [EUR]"] + df["CO2 cost [EUR]"]
    df.loc["TOTAL"] = df.sum()
    df.loc["TOTAL", ["full-load hours [h]", "hours at max capacity [h]"]] = np.nan
    return df


def check(name, results, parts):
    d = dispatch_of(results, parts)
    supply = d[units].sum(axis=1) + d["unserved heat"]
    imbalance = (supply - q_tot).abs().max()
    unserved_MWh = d["unserved heat"].sum()
    print(f"{name:<34s} solver: {parts['status']:<8s} | max imbalance {imbalance:.1e} MW | "
          f"unserved heat {1000 * unserved_MWh:.4f} kWh/yr")
    assert imbalance < 1e-4, "Heat balance violated"
    assert unserved_MWh < 1e-3, "Demand not met with the fixed capacities"
    return d

print("Model functions defined.")

## 4. Reference run: Step 4 with fixed capacities (consistency check)
Before adding emission costs, the Step 4 case is solved again with the capacities fixed. Its operating cost must equal the Step 4 result: this shows that the fixed-capacity model is the same energy system as Step 4, so any later change comes from the CO₂ cost alone.

In [ ]:
m_ref, r_ref, p_ref = solve_dispatch(emission_factors, co2_price=0.0)
dispatch_ref = check("Reference (Step 4, no CO2 cost)", r_ref, p_ref)
summary_ref = evaluate(r_ref, p_ref, emission_factors)

op_ref = summary_ref.loc["TOTAL", "operating cost [EUR]"]
op_s4 = summary_s4.loc["TOTAL", "operating cost [EUR]"] if "operating cost [EUR]" in summary_s4 else \
        summary_s4.loc["TOTAL", "fuel/electricity cost [EUR]"] + summary_s4.loc["TOTAL", "var O&M cost [EUR]"]
print(f"Operating cost: reference {op_ref:,.1f} EUR/yr vs Step 4 {op_s4:,.1f} EUR/yr "
      f"(difference {100 * (op_ref / op_s4 - 1):+.4f} %)")
assert abs(op_ref / op_s4 - 1) < 1e-3, "Reference run does not reproduce Step 4"
print("✅ The fixed-capacity model reproduces Step 4")

## 5. Scenario A (cost + emissions) and scenario B (emissions only)
- **A:** real fuel and electricity prices and variable O&M, plus 134 €/tCO₂ on every tonne.
- **B:** all fuel, electricity and variable O&M costs set to 0; only the CO₂ cost remains, so the dispatch is driven purely by emissions.

Each run is checked: heat balance in every hour, solver status, and unserved heat ≈ 0.

In [ ]:
m_A, r_A, p_A = solve_dispatch(emission_factors, co2_price=CARBON_PRICE, include_fuel=True, include_om=True)
m_B, r_B, p_B = solve_dispatch(emission_factors, co2_price=CARBON_PRICE, include_fuel=False, include_om=False)
dispatch_A = check("Scenario A (cost + CO2)", r_A, p_A)
dispatch_B = check("Scenario B (CO2 only)", r_B, p_B)

summary_A = evaluate(r_A, p_A, emission_factors)
summary_B = evaluate(r_B, p_B, emission_factors)
print("✅ Demand + losses met in all 8760 hours in both scenarios")

## 6. Results per unit
Costs are evaluated at the **real** prices in all three runs, so they can be compared directly: in scenario B the optimisation ignored fuel and O&M costs, but the system still has to pay them. The CO₂ cost is shown at 134 €/t in every run, including the reference, to show what the Step 4 dispatch would cost under the carbon price.

In [ ]:
show = ["heat [MWh]", "share of heat [%]", "full-load hours [h]", "hours at max capacity [h]",
        "operating cost [EUR]", "CO2 [t]", "CO2 cost [EUR]", "economic cost + CO2 cost [EUR]"]
for name, s in [("Reference (Step 4)", summary_ref), ("Scenario A: cost + CO2", summary_A),
                ("Scenario B: CO2 only", summary_B)]:
    print(name); display(s[show].round(2))

## 7. Dispatch over the year
### 7.1 Scenarios A and B, whole year and duration curves
✏️ Compare with the Step 4 dispatch (`results/step4_loss/dispatch.png`): which units move up or down in the merit order?

In [ ]:
for name, d, fname in [("Scenario A – cost + CO2 (134 EUR/t)", dispatch_A, "dispatch_A.png"),
                       ("Scenario B – CO2 cost only (134 EUR/t)", dispatch_B, "dispatch_B.png")]:
    plot_dispatch(d[units + ["demand"]], f"Heat production by unit – Step 5, {name}")
    plt.savefig(OUT / fname, dpi=150, bbox_inches="tight"); plt.show()

### 7.2 One winter week, hour by hour: Step 4 vs A vs B
The base load is the same in all three runs (the heat pumps run at full capacity); the differences are in **which peak unit** covers the hours above the heat pumps' capacity.

In [ ]:
start, end = "2019-01-14", "2019-01-20"
runs = [("Step 4", dispatch_ref), ("A: cost + CO2", dispatch_A), ("B: CO2 only", dispatch_B)]
fig, axs = plt.subplots(3, 1, figsize=(14, 11), sharex=True, sharey=True)
for ax, (name, d) in zip(axs, runs):
    w = d.loc[start:end]
    ax.stackplot(w.index, *[w[u] for u in units], labels=units, colors=[UNIT_COLORS[u] for u in units])
    ax.plot(w.index, w["demand"], "k", lw=1.2, label="demand + losses")
    ax.set_title(f"Hourly dispatch, winter week – {name}"); ax.set_ylabel("MW"); ax.grid(alpha=0.3)
axs[-1].legend(ncol=6, loc="upper center", bbox_to_anchor=(0.5, -0.15), fontsize=9, frameon=False)
plt.tight_layout(); plt.savefig(OUT / "dispatch_winter_week_comparison.png", dpi=150, bbox_inches="tight"); plt.show()

### 7.3 Monthly heat production by unit

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4.2), sharey=True)
for ax, (name, d) in zip(axs, runs):
    m = d[units].resample("MS").sum(); m.index = m.index.strftime("%b")
    m.plot.bar(stacked=True, ax=ax, width=0.8, color=[UNIT_COLORS[u] for u in units], legend=False)
    ax.set_title(name); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=90); ax.grid(axis="y", alpha=0.3)
axs[0].set_ylabel("Heat produced [MWh/month]")
handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=5, loc="lower center", bbox_to_anchor=(0.5, -0.06), frameon=False)
plt.tight_layout(); plt.savefig(OUT / "dispatch_monthly_comparison.png", dpi=150, bbox_inches="tight"); plt.show()

### 7.4 Hours at maximum capacity
Hours per year in which each unit runs at ≥ 99.9 % of its (fixed) capacity.

In [ ]:
hours_at_cap = pd.DataFrame({name: s.loc[units, "hours at max capacity [h]"]
                             for name, s in [("Step 4", summary_ref), ("A: cost + CO2", summary_A),
                                             ("B: CO2 only", summary_B)]}).astype(int)
display(hours_at_cap)

## 8. Comparison: Step 4 vs scenario A vs scenario B
⚠️ The two heat pumps are identical, so the split of heat between them is not unique. The figures group them as "heat pumps (P0 + P4)".

In [ ]:
HP_GROUP = "heat pumps (P0 + P4)"
group_of = {u: (HP_GROUP if tech.loc[u, "technology_type"] == "heat_pump" else u) for u in units}
groups = list(dict.fromkeys(group_of.values()))
GROUP_COLORS = {**UNIT_COLORS, HP_GROUP: UNIT_COLORS["heat pump"]}

def group_sum(series):
    return series[units].groupby(lambda u: group_of[u], sort=False).sum()[groups]

summaries = {"Step 4": summary_ref, "A: cost + CO2": summary_A, "B: CO2 only": summary_B}

# System totals
rows = ["heat [MWh]", "CO2 [t]", "fuel/electricity cost [EUR]", "var O&M cost [EUR]", "operating cost [EUR]",
        "annualised capex [EUR]", "economic cost [EUR]", "CO2 cost [EUR]", "economic cost + CO2 cost [EUR]"]
system = pd.DataFrame({k: s.loc["TOTAL", rows] for k, s in summaries.items()}).astype(float)
system.loc["CO2 intensity [kg/MWh delivered]"] = 1000 * system.loc["CO2 [t]"] / data["heat demand"].sum()
system.loc["economic cost per MWh delivered [EUR/MWh]"] = system.loc["economic cost [EUR]"] / data["heat demand"].sum()
for k in ["A: cost + CO2", "B: CO2 only"]:
    system[f"{k} vs Step 4 [%]"] = 100 * (system[k] / system["Step 4"] - 1)
display(system.round(2))

# Heat and CO2 per unit group
heat_g = pd.DataFrame({k: group_sum(s["heat [MWh]"]) for k, s in summaries.items()})
co2_g = pd.DataFrame({k: group_sum(s["CO2 [t]"]) for k, s in summaries.items()})
print("Annual heat per unit group [MWh]:"); display(heat_g.round(1))
print("Annual CO2 per unit group [t]:"); display(co2_g.round(2))

### 8.1 Graphical comparison
1. **Heat per unit:** how much each unit (group) produces in Step 4, A and B.
2. **Emissions per unit:** where the CO₂ comes from in each run.
3. **Economic vs environmental trade-off:** annual economic cost (real prices, incl. CAPEX) against annual CO₂ emissions. Lower left is better on both; a move left and up means emissions are cut at a cost.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(15, 4.8))
x = np.arange(len(groups)); w = 0.27
for i, k in enumerate(summaries):
    axs[0].bar(x + (i - 1) * w, heat_g[k], width=w, color=SCEN_COLORS[k], label=k)
    axs[1].bar(x + (i - 1) * w, co2_g[k], width=w, color=SCEN_COLORS[k], label=k)
for ax, ylabel, title in [(axs[0], "Heat produced [MWh/yr]", "Annual heat per unit"),
                          (axs[1], "CO2 emitted [t/yr]", "Annual CO2 emissions per unit")]:
    ax.set_xticks(x, groups, fontsize=9); ax.set_ylabel(ylabel); ax.set_title(title); ax.grid(axis="y", alpha=0.3)
axs[0].set_yscale("log"); axs[0].set_ylabel("Heat produced [MWh/yr] (log scale)")
axs[1].legend(loc="upper center", bbox_to_anchor=(-0.1, -0.12), ncol=3, frameon=False)
plt.tight_layout(); plt.savefig(OUT / "comparison_heat_co2_per_unit.png", dpi=150, bbox_inches="tight"); plt.show()

## 9. Sensitivity analyses
### 9.1 Carbon price in scenario A: 0 – 500 €/t
Scenario A is solved again for a range of carbon prices. This shows at which price the dispatch changes and how much CO₂ each price removes. 134 €/t is the main case.

In [ ]:
PRICES = [0, 25, 50, 100, 134, 200, 300, 500]
sweep_rows, sweep_heat = [], {}
for price in PRICES:
    m_s, r_s, p_s = solve_dispatch(emission_factors, co2_price=price)
    check(f"Scenario A at {price:>3d} EUR/t", r_s, p_s)
    s = evaluate(r_s, p_s, emission_factors, co2_price=price)
    sweep_heat[price] = group_sum(s["heat [MWh]"])
    sweep_rows.append({"carbon price [EUR/t]": price,
                       "CO2 [t]": s.loc["TOTAL", "CO2 [t]"],
                       "economic cost [EUR]": s.loc["TOTAL", "economic cost [EUR]"],
                       "CO2 cost [EUR]": s.loc["TOTAL", "CO2 cost [EUR]"]})
sweep = pd.DataFrame(sweep_rows).set_index("carbon price [EUR/t]")
sweep_heat = pd.DataFrame(sweep_heat).T
sweep_heat.index.name = "carbon price [EUR/t]"
display(sweep.round(1)); display(sweep_heat.round(1))

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(17, 4.6))

# (a) heat from the peak units vs carbon price (heat pumps are at full capacity and change little)
ax = axs[0]
for g in groups:
    if g != HP_GROUP:
        ax.plot(sweep_heat.index, sweep_heat[g], "o-", color=GROUP_COLORS[g], lw=2, ms=5, label=g)
ax.axvline(CARBON_PRICE, color="k", ls=":", lw=1)
ax.set_xlabel("Carbon price [EUR/tCO2]"); ax.set_ylabel("Heat produced [MWh/yr]")
ax.set_title("Peak units: annual heat vs carbon price"); ax.legend(fontsize=9); ax.grid(alpha=0.3)

# (b) CO2 vs carbon price
ax = axs[1]
ax.plot(sweep.index, sweep["CO2 [t]"], "o-", color=SCEN_COLORS["A: cost + CO2"], lw=2, ms=5)
ax.axvline(CARBON_PRICE, color="k", ls=":", lw=1)
ax.set_xlabel("Carbon price [EUR/tCO2]"); ax.set_ylabel("CO2 emitted [t/yr]")
ax.set_title("Annual emissions vs carbon price"); ax.grid(alpha=0.3)

# (c) trade-off: economic cost vs CO2, with the three main runs
ax = axs[2]
ax.plot(sweep["CO2 [t]"], sweep["economic cost [EUR]"] / 1000, "-", color="#BDBDBD", lw=1.5, zorder=1,
        label="scenario A, carbon price 0–500 EUR/t")
for price in [0, 134, 500]:
    ax.annotate(f"{price} EUR/t", (sweep.loc[price, "CO2 [t]"], sweep.loc[price, "economic cost [EUR]"] / 1000),
                textcoords="offset points", xytext=(6, -12), fontsize=8, color="#616161")
for k, s in summaries.items():
    ax.scatter(s.loc["TOTAL", "CO2 [t]"], s.loc["TOTAL", "economic cost [EUR]"] / 1000,
               s=90, color=SCEN_COLORS[k], zorder=3, label=k, edgecolor="white", linewidth=1.5)
ax.set_xlabel("CO2 emitted [t/yr]"); ax.set_ylabel("Economic cost [kEUR/yr] (real prices, incl. CAPEX)")
ax.set_title("Trade-off: economic cost vs emissions"); ax.legend(fontsize=8, loc="best"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT / "sensitivity_carbon_price_tradeoff.png", dpi=150, bbox_inches="tight"); plt.show()

### 9.2 Biogenic CO₂ counted as zero
The same scenarios A and B, but with the CO₂ from wood pellets and biomethane set to zero, as in IPCC national inventories and the EU ETS. Only the electricity emissions are then priced. This tests how much the result depends on the accounting rule chosen in section 2.

Note: in scenario B the two fuel boilers then have **no variable cost at all**, so the solver has no reason to prefer one over the other between them; their split is arbitrary.

In [ ]:
bio_rows = {}
for name, fuel in [("A: cost + CO2", True), ("B: CO2 only", False)]:
    m_x, r_x, p_x = solve_dispatch(ef_biogenic_zero, co2_price=CARBON_PRICE, include_fuel=fuel, include_om=fuel)
    check(f"{name}, biogenic CO2 = 0", r_x, p_x)
    s_x = evaluate(r_x, p_x, emission_factors)        # emissions reported with biogenic CO2 counted
    s_x0 = evaluate(r_x, p_x, ef_biogenic_zero)       # ... and with the zero convention
    bio_rows[f"{name}, biogenic = 0"] = pd.concat([group_sum(s_x["heat [MWh]"]), pd.Series({
        "economic cost [EUR]": s_x.loc["TOTAL", "economic cost [EUR]"],
        "CO2 incl. biogenic [t]": s_x.loc["TOTAL", "CO2 [t]"],
        "CO2 fossil/electricity only [t]": s_x0.loc["TOTAL", "CO2 [t]"]})])
for name, s in [("A: cost + CO2", summary_A), ("B: CO2 only", summary_B)]:
    bio_rows[f"{name}, biogenic counted (main)"] = pd.concat([group_sum(s["heat [MWh]"]), pd.Series({
        "economic cost [EUR]": s.loc["TOTAL", "economic cost [EUR]"],
        "CO2 incl. biogenic [t]": s.loc["TOTAL", "CO2 [t]"],
        "CO2 fossil/electricity only [t]": float((pd.Series({u: s.loc[u, "CO2 [t]"] for u in units})
                                                 [tech["input"] == "electricity"]).sum())})])
biogenic_sensitivity = pd.DataFrame(bio_rows)
display(biogenic_sensitivity.round(2))

## 10. Save results
CSV files and one Excel file with all tables for the report. Figures were saved above.

In [ ]:
for name, s in [("reference", summary_ref), ("A_cost_co2", summary_A), ("B_co2_only", summary_B)]:
    s.to_csv(OUT / f"unit_summary_{name}.csv")
pd.concat({"Step 4": dispatch_ref, "A": dispatch_A, "B": dispatch_B}, axis=1).to_csv(OUT / "dispatch_all.csv")
system.to_csv(OUT / "system_comparison.csv")
merit.to_csv(OUT / "merit_order.csv")
emission_factors.to_csv(OUT / "emission_factors.csv")
sweep.join(sweep_heat).to_csv(OUT / "sensitivity_carbon_price.csv")
biogenic_sensitivity.to_csv(OUT / "sensitivity_biogenic_co2.csv")
hours_at_cap.to_csv(OUT / "hours_at_capacity.csv")

excel_file = OUT / "step5_emissions_results.xlsx"
try:
    with pd.ExcelWriter(excel_file) as writer:
        system.round(3).to_excel(writer, sheet_name="System comparison")
        heat_g.round(3).to_excel(writer, sheet_name="Heat per unit group")
        co2_g.round(3).to_excel(writer, sheet_name="CO2 per unit group")
        summary_ref.round(3).to_excel(writer, sheet_name="Units Step4 reference")
        summary_A.round(3).to_excel(writer, sheet_name="Units scenario A")
        summary_B.round(3).to_excel(writer, sheet_name="Units scenario B")
        hours_at_cap.to_excel(writer, sheet_name="Hours at capacity")
        merit.round(3).to_excel(writer, sheet_name="Merit order")
        emission_factors.to_excel(writer, sheet_name="Emission factors")
        sweep.join(sweep_heat).round(3).to_excel(writer, sheet_name="Sens carbon price")
        biogenic_sensitivity.round(3).to_excel(writer, sheet_name="Sens biogenic CO2")
        for name, d in runs:
            d.resample("MS").sum().round(3).to_excel(writer, sheet_name=f"Monthly heat {name[:1]}")
    print("Saved", excel_file)
except ImportError:
    print("Excel file skipped: run  %pip install openpyxl  once, then rerun this cell.")
print("Saved CSV files and figures to", OUT)

## 11. Key numbers for the answers
Printed from the results above, so the answers in section 12 can be checked against the current run.

In [ ]:
print(f"Carbon price {CARBON_PRICE:.0f} EUR/t | biogenic CO2 counted | capacities fixed at Step 4 ({cap.sum():.3f} MW)\n")
for k in summaries:
    print(f"{k:<14s} CO2 {system.loc['CO2 [t]', k]:6.1f} t/yr | economic cost {system.loc['economic cost [EUR]', k]:9,.0f} EUR/yr | "
          f"CO2 cost {system.loc['CO2 cost [EUR]', k]:8,.0f} EUR/yr | total {system.loc['economic cost + CO2 cost [EUR]', k]:9,.0f} EUR/yr")
print("\nHeat per unit group [MWh/yr]  (Step 4 -> A -> B):")
for g in groups:
    print(f"  {g:<24s} {heat_g.loc[g, 'Step 4']:8.1f} -> {heat_g.loc[g, 'A: cost + CO2']:8.1f} -> {heat_g.loc[g, 'B: CO2 only']:8.1f}")
print("\nCO2 per unit group [t/yr]  (Step 4 -> A -> B):")
for g in groups:
    print(f"  {g:<24s} {co2_g.loc[g, 'Step 4']:7.2f} -> {co2_g.loc[g, 'A: cost + CO2']:7.2f} -> {co2_g.loc[g, 'B: CO2 only']:7.2f}")
dA = system.loc["CO2 [t]", "A: cost + CO2"] - system.loc["CO2 [t]", "Step 4"]
cA = system.loc["economic cost [EUR]", "A: cost + CO2"] - system.loc["economic cost [EUR]", "Step 4"]
if dA < -1e-6:
    print(f"\nScenario A: {-dA:.2f} t/yr less CO2 for {cA:+,.0f} EUR/yr extra economic cost "
          f"= {cA / -dA:,.0f} EUR per tonne avoided")

## 12. Questions to answer
✏️ Write the answers with the numbers from section 11.

1. **In scenario A, which units become less competitive when emission costs are included?** *(section 2 merit order, sections 6–8)*
   Hint: compare each unit's CO₂ cost per MWh heat with the gap to the next unit in the merit order. With biogenic CO₂ counted, the fuel boilers carry by far the largest CO₂ cost per MWh.

2. **In scenario B, which low-carbon units dominate operation?** *(sections 6–7)*
   Hint: with only CO₂ priced, the order follows tCO₂ per MWh heat. Which unit has the lowest, and why does the COP matter?

3. **What is the trade-off between economic and environmental outcomes, considering Step 4 and both Step 5 scenarios?** *(section 8 system table, section 9.1 trade-off figure)*
   Hint: how much CO₂ does each scenario avoid compared with Step 4, and at what extra economic cost (€ per tonne avoided)?

Points for the discussion (grade A):
- **Fixed capacities limit the effect.** The heat pumps already run at full capacity in Step 4, so a carbon price can only reshuffle the peak units. Larger reductions would need a different plant, i.e. re-optimised capacities (Step 3/4 approach), which this step deliberately does not do.
- **Accounting matters (section 9.2).** With biogenic CO₂ = 0, as in IPCC inventories and the EU ETS, wood and biomethane appear emission-free and the conclusions for the boilers change.
- **Limitations:** direct emissions only (no upstream or plant manufacture), constant COP, average (not marginal) electricity emission factor, spot prices without taxes and grid fees.